# 💬 Phi-3 Mini — Smart Text Summarizer
### Paste any long text/article and Phi-3 will summarize it for you!
#### Supports: Short Summary, Bullet Points, and Detailed Summary modes
> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU` before running!

In [ ]:
# ✅ Cell 1 — Install Dependencies
!pip install transformers==4.41.2 accelerate -q
print('✅ All libraries installed!')

In [ ]:
# ✅ Cell 2 — Check GPU
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f'\n✅ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    print('\n⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ✅ Cell 3 — All Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import textwrap

print('✅ All imports done!')
print(f'🖥️ Device: {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}')

In [ ]:
# ✅ Cell 4 — Load Phi-3 Model
model_id = 'microsoft/Phi-3-mini-4k-instruct'

print('⏳ Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print('⏳ Loading model... please wait (2-5 mins first time)')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager'
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)

print('✅ Model loaded and ready!')

In [ ]:
# ✅ Cell 5 — Summarizer Functions

def summarize(text: str, mode: str = 'short') -> str:
    """
    Summarize text using Phi-3.
    mode options:
      'short'   → 2-3 sentence summary
      'bullets' → key points as bullet list
      'detailed'→ full detailed summary
    """

    if mode == 'short':
        system_prompt = """You are a text summarization assistant.
Summarize the given text in 2-3 clear and concise sentences.
Capture the most important points only.
Do not add any extra commentary."""

    elif mode == 'bullets':
        system_prompt = """You are a text summarization assistant.
Summarize the given text as a clean bullet point list.
Extract the 5 most important key points.
Format each point starting with • symbol.
Do not add any extra commentary."""

    elif mode == 'detailed':
        system_prompt = """You are a text summarization assistant.
Write a detailed summary of the given text.
Cover all important points, facts, and conclusions.
Keep it well structured and easy to read.
Do not add any extra commentary."""

    else:
        return 'Invalid mode. Use: short, bullets, or detailed'

    # Truncate text if too long (Phi-3 has 4k context limit)
    max_input_chars = 2500
    if len(text) > max_input_chars:
        text = text[:max_input_chars] + '...'
        print(f'⚠️ Text was too long, truncated to {max_input_chars} characters')

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'Please summarize this text:\n\n{text}'}
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    return output[0]['generated_text'][len(prompt):].strip()


def summarize_all(text: str):
    """Run all 3 summary modes at once."""
    print('\n⏳ Generating Short Summary...')
    short = summarize(text, mode='short')

    print('⏳ Generating Bullet Points...')
    bullets = summarize(text, mode='bullets')

    print('⏳ Generating Detailed Summary...')
    detailed = summarize(text, mode='detailed')

    print('\n' + '='*60)
    print('📝 SHORT SUMMARY (2-3 sentences):')
    print('='*60)
    print(short)

    print('\n' + '='*60)
    print('🔵 KEY BULLET POINTS:')
    print('='*60)
    print(bullets)

    print('\n' + '='*60)
    print('📄 DETAILED SUMMARY:')
    print('='*60)
    print(detailed)

print('✅ Summarizer functions ready!')

In [ ]:
# ✅ Cell 6 — Choose Your Mode
# Options:
#   summarize(text, mode='short')    → 2-3 sentence summary
#   summarize(text, mode='bullets')  → bullet point summary
#   summarize(text, mode='detailed') → full detailed summary
#   summarize_all(text)              → all 3 modes at once!

# 📌 PASTE YOUR TEXT HERE:
my_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by animals including humans.
AI research has been defined as the field of study of intelligent agents,
which refers to any system that perceives its environment and takes actions
that maximize its chance of achieving its goals. The term artificial intelligence
had previously been used to describe machines that mimic and display human
cognitive skills associated with the human mind, such as learning and problem-solving.
This definition has since been rejected by major AI researchers who now describe AI
in terms of rationality and acting rationally, which does not limit how intelligence
can be articulated. AI applications include advanced web search engines, recommendation
systems, understanding human speech, self-driving cars, generative tools,
automated decision-making and competing at the highest level in strategic game systems.
As machines become increasingly capable, tasks considered to require intelligence
are often removed from the definition of AI, a phenomenon known as the AI effect.
For instance, optical character recognition is frequently excluded from things
considered to be AI, having become a routine technology.
"""

# Run all 3 summary modes at once:
summarize_all(my_text)

In [ ]:
# ✅ Cell 7 — Interactive Mode (paste & summarize any text!)
print('=' * 60)
print('   💬 Phi-3 Mini — Interactive Text Summarizer')
print('=' * 60)
print('\n📌 How to use:')
print('   1. Paste your text')
print('   2. Type END on a new line when done')
print('   3. Choose summary mode')
print('   Type quit to exit')
print('=' * 60)

while True:
    print('\n📋 Paste your text below (type END on new line when done):')
    lines = []
    while True:
        line = input()
        if line.strip().lower() == 'quit':
            print('👋 Goodbye!')
            exit()
        if line.strip().upper() == 'END':
            break
        lines.append(line)

    text = '\n'.join(lines).strip()
    if not text:
        print('⚠️ No text entered!')
        continue

    print('\n🎯 Choose summary mode:')
    print('   1 → Short (2-3 sentences)')
    print('   2 → Bullet Points')
    print('   3 → Detailed')
    print('   4 → All 3 modes')
    choice = input('\nEnter choice (1/2/3/4): ').strip()

    if choice == '1':
        print('\n⏳ Summarizing...')
        result = summarize(text, mode='short')
        print(f'\n📝 SHORT SUMMARY:\n{result}')
    elif choice == '2':
        print('\n⏳ Summarizing...')
        result = summarize(text, mode='bullets')
        print(f'\n🔵 BULLET POINTS:\n{result}')
    elif choice == '3':
        print('\n⏳ Summarizing...')
        result = summarize(text, mode='detailed')
        print(f'\n📄 DETAILED SUMMARY:\n{result}')
    elif choice == '4':
        summarize_all(text)
    else:
        print('⚠️ Invalid choice! Enter 1, 2, 3, or 4')